In [10]:
import os
import pandas as pd
import polars as pl
from pathlib import Path
from typing import Optional
import pickle

# MF多任务低频量价模型 - 日K转周K/月K

研报03部分：基于多任务学习的低频量价模型

目标：将2013-2026年沪深300成分股日K数据合成为周K、月K

核心逻辑：
- open: 周期内第一个交易日的开盘价
- high: 周期内最高价
- low: 周期内最低价
- close: 周期内最后一个交易日的收盘价
- volume: 周期内成交量总和
- turnover: 周期内成交额总和
- vwap: 周期内成交额/成交量
- open_interest: 周期最后一个交易日的持仓量

In [11]:
# ============ 配置参数 ============
LAB_PATH = Path(r"D:\\Aquant project\\MF\\lab2")
DAILY_DIR = LAB_PATH / "daily"      # 日K数据目录
WEEKLY_DIR = LAB_PATH / "weekly"    # 周K输出目录
MONTHLY_DIR = LAB_PATH / "monthly"  # 月K输出目录

# 确保输出目录存在
WEEKLY_DIR.mkdir(parents=True, exist_ok=True)
MONTHLY_DIR.mkdir(parents=True, exist_ok=True)

# 时间范围
START_DATE = "2013-01-01"
END_DATE = "2026-03-27"

# 内存控制：每批处理的股票数量
BATCH_SIZE = 50

print(f"日K目录: {DAILY_DIR}")
print(f"周K目录: {WEEKLY_DIR}")
print(f"月K目录: {MONTHLY_DIR}")

日K目录: D:\Aquant project\MF\lab2\daily
周K目录: D:\Aquant project\MF\lab2\weekly
月K目录: D:\Aquant project\MF\lab2\monthly


## Step 1: 构建交易日历

从日K数据中提取所有不重复的交易日期，生成统一的交易日历。
这是处理数据参差不齐的关键：以实际存在的交易日为准。

In [12]:
def build_trading_calendar(daily_dir: Path, start_date: str, end_date: str) -> pd.DataFrame:
    """
    从日K数据中提取交易日历，并生成周/月标签
    
    策略：
    1. 扫描所有日K文件，提取datetime列
    2. 合并去重得到完整交易日历
    3. 标记每个交易日所属的交易周、交易月
    """
    import polars as pl
    
    start_dt = pd.Timestamp(start_date)
    end_dt = pd.Timestamp(end_date)
    
    all_dates = set()
    
    # 获取所有parquet文件
    data_files = list(daily_dir.glob("*.parquet"))
    print(f"发现 {len(data_files)} 个日K数据文件")
    
    for i, fpath in enumerate(data_files):
        if i % 100 == 0:
            print(f"  扫描进度: {i}/{len(data_files)}")
        try:
            # 只读取datetime列，节省内存
            df = pl.read_parquet(fpath, columns=['datetime'])
            # 转换为pandas获取date
            dates = df['datetime'].dt.date().to_list()
            all_dates.update(dates)
        except Exception as e:
            print(f"  跳过 {fpath.name}: {e}")
            continue
    
    # 构建交易日历DataFrame
    calendar = pd.DataFrame({'datetime': sorted(all_dates)})
    calendar['datetime'] = pd.to_datetime(calendar['datetime'])
    calendar = calendar[(calendar['datetime'] >= start_dt) & (calendar['datetime'] <= end_dt)]
    calendar = calendar.sort_values('datetime').reset_index(drop=True)
    
    print(f"\n共提取 {len(calendar)} 个交易日")
    
    # 生成交易周标签 (ISO周: 年-周序号)
    calendar['year_week'] = calendar['datetime'].dt.isocalendar().year.astype(str) + '-' + calendar['datetime'].dt.isocalendar().week.astype(str).str.zfill(2)
    
    # 生成交易月标签 (年-月)
    calendar['year_month'] = calendar['datetime'].dt.strftime('%Y-%m')
    
    # 计算每周/每月的第一个和最后一个交易日
    weekly_boundaries = calendar.groupby('year_week')['datetime'].agg(['min', 'max']).reset_index()
    weekly_boundaries.columns = ['year_week', 'week_start', 'week_end']
    
    monthly_boundaries = calendar.groupby('year_month')['datetime'].agg(['min', 'max']).reset_index()
    monthly_boundaries.columns = ['year_month', 'month_start', 'month_end']
    
    # 合并回日历表
    calendar = calendar.merge(weekly_boundaries, on='year_week', how='left')
    calendar = calendar.merge(monthly_boundaries, on='year_month', how='left')
    
    return calendar

In [13]:
# 执行：构建或加载交易日历
calendar_path = LAB_PATH / "trading_calendar.pkl"

if calendar_path.exists():
    print("加载已存在的交易日历...")
    with open(calendar_path, 'rb') as f:
        calendar = pickle.load(f)
else:
    print("构建交易日历...")
    calendar = build_trading_calendar(DAILY_DIR, START_DATE, END_DATE)
    with open(calendar_path, 'wb') as f:
        pickle.dump(calendar, f)
    print(f"交易日历已保存到 {calendar_path}")

print(f"\n交易日历样例:")
print(calendar.head(10))

加载已存在的交易日历...

交易日历样例:
    datetime year_week year_month week_start   week_end month_start  month_end
0 2013-01-04   2013-01    2013-01 2013-01-04 2013-01-04  2013-01-04 2013-01-31
1 2013-01-07   2013-02    2013-01 2013-01-07 2013-01-11  2013-01-04 2013-01-31
2 2013-01-08   2013-02    2013-01 2013-01-07 2013-01-11  2013-01-04 2013-01-31
3 2013-01-09   2013-02    2013-01 2013-01-07 2013-01-11  2013-01-04 2013-01-31
4 2013-01-10   2013-02    2013-01 2013-01-07 2013-01-11  2013-01-04 2013-01-31
5 2013-01-11   2013-02    2013-01 2013-01-07 2013-01-11  2013-01-04 2013-01-31
6 2013-01-14   2013-03    2013-01 2013-01-14 2013-01-18  2013-01-04 2013-01-31
7 2013-01-15   2013-03    2013-01 2013-01-14 2013-01-18  2013-01-04 2013-01-31
8 2013-01-16   2013-03    2013-01 2013-01-14 2013-01-18  2013-01-04 2013-01-31
9 2013-01-17   2013-03    2013-01 2013-01-14 2013-01-18  2013-01-04 2013-01-31


## Step 2: 单股票K线合成函数

输入：单只股票的日K数据 (polars DataFrame)
输出：周K、月K数据

数据列：datetime, open, high, low, close, volume, turnover, open_interest

In [14]:
def resample_daily_to_weekly_monthly(daily_df: pl.DataFrame, calendar: pd.DataFrame) -> tuple[pl.DataFrame, pl.DataFrame]:
    """
    将单只股票的日K数据合成为周K和月K
    
    Parameters:
        daily_df: polars DataFrame, 列: [datetime, open, high, low, close, volume, turnover, open_interest]
        calendar: pandas DataFrame, 交易日历
    
    Returns: (weekly_df, monthly_df) 两个polars DataFrame
    """
    import polars as pl
    
    # 转换为pandas以便与calendar合并
    df = daily_df.to_pandas()
    df['datetime'] = pd.to_datetime(df['datetime'])
    
    # 与交易日历合并，获取周/月标签
    merge_cols = ['datetime', 'year_week', 'year_month', 'week_start', 'week_end', 'month_start', 'month_end']
    df = df.merge(calendar[merge_cols], on='datetime', how='inner')
    
    if len(df) == 0:
        return pl.DataFrame(), pl.DataFrame()
    
    # ============ 合成周K ============
    weekly_list = []
    
    for year_week, group in df.groupby('year_week'):
        if len(group) == 0:
            continue
        
        group = group.sort_values('datetime')
        
        # 计算vwap: 总成交额 / 总成交量
        total_turnover = group['turnover'].sum()
        total_volume = group['volume'].sum()
        vwap = total_turnover / total_volume if total_volume > 0 else 0
        
        weekly_bar = {
            'datetime': group['week_end'].iloc[0],  # 用周期最后一个交易日作为代表日期
            'year_week': year_week,
            'open': group['open'].iloc[0],
            'high': group['high'].max(),
            'low': group['low'].min(),
            'close': group['close'].iloc[-1],
            'volume': total_volume,
            'turnover': total_turnover,
            'vwap': vwap,
            'open_interest': group['open_interest'].iloc[-1],  # 取最后一天的持仓量
            'trade_days': len(group)
        }
        weekly_list.append(weekly_bar)
    
    weekly_df = pl.DataFrame(weekly_list) if weekly_list else pl.DataFrame()
    
    # ============ 合成月K ============
    monthly_list = []
    
    for year_month, group in df.groupby('year_month'):
        if len(group) == 0:
            continue
        
        group = group.sort_values('datetime')
        
        total_turnover = group['turnover'].sum()
        total_volume = group['volume'].sum()
        vwap = total_turnover / total_volume if total_volume > 0 else 0
        
        monthly_bar = {
            'datetime': group['month_end'].iloc[0],
            'year_month': year_month,
            'open': group['open'].iloc[0],
            'high': group['high'].max(),
            'low': group['low'].min(),
            'close': group['close'].iloc[-1],
            'volume': total_volume,
            'turnover': total_turnover,
            'vwap': vwap,
            'open_interest': group['open_interest'].iloc[-1],
            'trade_days': len(group)
        }
        monthly_list.append(monthly_bar)
    
    monthly_df = pl.DataFrame(monthly_list) if monthly_list else pl.DataFrame()
    
    return weekly_df, monthly_df

## Step 3: 批量处理所有股票

内存优化策略：
1. 使用polars惰性读取，减少内存占用
2. 逐个处理，不累积所有股票数据
3. 直接保存为parquet格式

In [15]:
def process_all_stocks(daily_dir: Path, weekly_dir: Path, monthly_dir: Path, 
                       calendar: pd.DataFrame) -> dict:
    """
    批量处理所有股票的日K数据，逐只显示详细进展
    """
    from datetime import datetime
    
    data_files = list(daily_dir.glob("*.parquet"))
    
    stats = {
        'total': len(data_files),
        'success': 0,
        'failed': 0,
        'weekly_bars': 0,
        'monthly_bars': 0
    }
    
    print(f"\n{'='*60}")
    print(f"开始处理 {len(data_files)} 只股票")
    print(f"{'='*60}\n")
    
    for i, fpath in enumerate(data_files, 1):
        vt_symbol = fpath.stem
        start_time = datetime.now()
        
        print(f"[{i}/{len(data_files)}] {start_time.strftime('%H:%M:%S')} 开始处理: {vt_symbol}")
        
        try:
            # 读取日K数据
            daily_df = pl.read_parquet(fpath)
            
            # 合成周K、月K
            weekly_df, monthly_df = resample_daily_to_weekly_monthly(daily_df, calendar)
            
            # 保存周K并统计
            weekly_count = len(weekly_df)
            if weekly_count > 0:
                weekly_df.write_parquet(weekly_dir / f"{vt_symbol}.parquet", compression='zstd')
                stats['weekly_bars'] += weekly_count
            
            # 保存月K并统计
            monthly_count = len(monthly_df)
            if monthly_count > 0:
                monthly_df.write_parquet(monthly_dir / f"{vt_symbol}.parquet", compression='zstd')
                stats['monthly_bars'] += monthly_count
            
            stats['success'] += 1
            
            end_time = datetime.now()
            elapsed = (end_time - start_time).total_seconds()
            print(f"  ✓ 成功 ({elapsed:.2f}s) - 周K: {weekly_count}条, 月K: {monthly_count}条")
            
        except Exception as e:
            stats['failed'] += 1
            end_time = datetime.now()
            elapsed = (end_time - start_time).total_seconds()
            print(f"  ✗ 失败 ({elapsed:.2f}s) - 原因: {e}")
    
    # 最终统计
    print(f"\n{'='*60}")
    print(f"处理完成!")
    print(f"{'='*60}")
    print(f"总股票数: {stats['total']}")
    print(f"成功: {stats['success']}")
    print(f"失败: {stats['failed']}")
    print(f"周K总数: {stats['weekly_bars']}")
    print(f"月K总数: {stats['monthly_bars']}")
    
    return stats

In [17]:
# 执行批量处理
stats = process_all_stocks(DAILY_DIR, WEEKLY_DIR, MONTHLY_DIR, calendar)


开始处理 715 只股票

[1/715] 17:01:40 开始处理: 000001.SZSE
  ✓ 成功 (0.40s) - 周K: 678条, 月K: 159条
[2/715] 17:01:40 开始处理: 000002.SZSE
  ✓ 成功 (0.32s) - 周K: 678条, 月K: 159条
[3/715] 17:01:40 开始处理: 000008.SZSE
  ✓ 成功 (0.29s) - 周K: 678条, 月K: 159条
[4/715] 17:01:41 开始处理: 000009.SZSE
  ✓ 成功 (0.31s) - 周K: 678条, 月K: 159条
[5/715] 17:01:41 开始处理: 000012.SZSE
  ✓ 成功 (0.30s) - 周K: 678条, 月K: 159条
[6/715] 17:01:41 开始处理: 000024.SZSE
  ✓ 成功 (0.09s) - 周K: 156条, 月K: 36条
[7/715] 17:01:41 开始处理: 000027.SZSE
  ✓ 成功 (0.30s) - 周K: 678条, 月K: 159条
[8/715] 17:01:42 开始处理: 000039.SZSE
  ✓ 成功 (0.28s) - 周K: 678条, 月K: 159条
[9/715] 17:01:42 开始处理: 000046.SZSE
  ✓ 成功 (0.24s) - 周K: 569条, 月K: 134条
[10/715] 17:01:42 开始处理: 000059.SZSE
  ✓ 成功 (0.31s) - 周K: 678条, 月K: 159条
[11/715] 17:01:42 开始处理: 000060.SZSE
  ✓ 成功 (0.36s) - 周K: 678条, 月K: 159条
[12/715] 17:01:43 开始处理: 000061.SZSE
  ✓ 成功 (0.30s) - 周K: 678条, 月K: 159条
[13/715] 17:01:43 开始处理: 000063.SZSE
  ✓ 成功 (0.30s) - 周K: 678条, 月K: 159条
[14/715] 17:01:43 开始处理: 000066.SZSE
  ✓ 成功 (0.30s) - 周K: 67

## Step 4: 验证结果

随机抽样检查几只股票的合成结果。

In [ ]:
def verify_resample(vt_symbol: str, daily_dir: Path, weekly_dir: Path, monthly_dir: Path):
    """验证单股票的K线合成结果"""
    
    # 读取原始日K
    daily_df = pl.read_parquet(daily_dir / f"{vt_symbol}.parquet").to_pandas()
    daily_df['datetime'] = pd.to_datetime(daily_df['datetime'])
    
    # 读取合成的周K、月K
    weekly_df = pl.read_parquet(weekly_dir / f"{vt_symbol}.parquet").to_pandas()
    monthly_df = pl.read_parquet(monthly_dir / f"{vt_symbol}.parquet").to_pandas()
    
    print(f"===== {vt_symbol} 验证结果 =====")
    print(f"日K数量: {len(daily_df)} ({daily_df['datetime'].min().date()} ~ {daily_df['datetime'].max().date()})")
    print(f"周K数量: {len(weekly_df)}")
    print(f"月K数量: {len(monthly_df)}")
    
    # 验证某一周的合成
    if len(weekly_df) > 0:
        sample = weekly_df.iloc[0]
        print(f"\n第1周 ({sample['datetime']}):")
        print(f"  open={sample['open']:.2f}, close={sample['close']:.2f}")
        print(f"  vwap={sample['vwap']:.2f} (验证: {sample['turnover']/sample['volume']:.2f})")
    
    # 验证某一月
    if len(monthly_df) > 0:
        sample = monthly_df.iloc[0]
        print(f"\n第1月 ({sample['datetime']}):")
        print(f"  volume={sample['volume']:.0f}")
        print(f"  trade_days={sample['trade_days']}")

# 获取已处理的股票进行验证
weekly_files = list(WEEKLY_DIR.glob("*.parquet"))
if weekly_files:
    samples = [f.stem for f in weekly_files[:3]]
    for sym in samples:
        verify_resample(sym, DAILY_DIR, WEEKLY_DIR, MONTHLY_DIR)
        print("\n" + "="*50 + "\n")

## 总结

生成的数据文件结构：
```
lab2/
├── daily/{vt_symbol}.parquet     # 原始日K
├── weekly/{vt_symbol}.parquet    # 合成周K
├── monthly/{vt_symbol}.parquet   # 合成月K
└── trading_calendar.pkl          # 交易日历缓存
```

周K/月K数据列：
- datetime: 周期最后一个交易日
- open/high/low/close: 周期开高低收
- volume/turnover: 周期总量
- vwap: 周期成交额/成交量
- open_interest: 周期最后一天的持仓量
- trade_days: 周期内实际交易日数量